# Retroactively Log Unfrozen Model to MLflow

This notebook logs the trained unfrozen model (saved to DBFS) to MLflow artifacts.

**Problem:** The `finetune_unfrozen.ipynb` notebook saved the model to DBFS but did NOT log it to MLflow artifacts.

**Solution:** Load the model from DBFS and log it to the existing MLflow run.

**Model location:** `/dbfs/FileStore/allen_brain_data/models/unfrozen`

In [0]:
# Cell 0 - Configuration

import mlflow
import os

# Path where the model was saved during training
MODEL_PATH = "/dbfs/FileStore/allen_brain_data/models/unfrozen"

# MLflow experiment (same as in training)
EXPERIMENT_NAME = "/Users/noel.nosse@grainger.com/histology-brain-segmentation"

# OPTION 1: Get the run ID from the training notebook output or MLflow UI
# After running the cells below to search for the run, you can uncomment and set this:
# RUN_ID = "your-run-id-here"
RUN_ID = None  # Will search for it if None

print(f"Model path: {MODEL_PATH}")
print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Run ID: {RUN_ID or 'Will search automatically'}")

Model path: /dbfs/FileStore/allen_brain_data/models/unfrozen
Experiment: /Users/noel.nosse@grainger.com/histology-brain-segmentation
Run ID: Will search automatically


In [0]:
# Cell 1 - Verify model exists on DBFS

import os

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Model not found at {MODEL_PATH}")

# List contents to verify
contents = os.listdir(MODEL_PATH)
print(f"Model directory contents ({len(contents)} files):")
for item in sorted(contents)[:20]:  # Show first 20 files
    size = os.path.getsize(os.path.join(MODEL_PATH, item))
    print(f"  {item:50s} {size:>15,} bytes")

if len(contents) > 20:
    print(f"  ... and {len(contents) - 20} more files")

# Check for required files
required_files = ["config.json", "model.safetensors", "preprocessor_config.json"]
missing = [f for f in required_files if f not in contents]
if missing:
    print(f"\n⚠️  WARNING: Missing expected files: {missing}")
else:
    print(f"\n✅ All required model files present")

Model directory contents (3 files):
  config.json                                                 63,854 bytes
  model.safetensors                                    1,368,529,704 bytes
  training_args.bin                                            5,777 bytes

⚠️  WARNING: Missing expected files: ['preprocessor_config.json']


In [0]:
# Cell 2 - Find the MLflow run (if RUN_ID not provided)

if RUN_ID is None:
    experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
    if experiment is None:
        raise ValueError(f"Experiment '{EXPERIMENT_NAME}' not found")
    
    # Search for runs with "unfrozen" in the name
    runs = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        filter_string="tags.mlflow.runName LIKE 'unfrozen%'",
        order_by=["start_time DESC"],
        max_results=10
    )
    
    if len(runs) == 0:
        raise ValueError("No runs found with 'unfrozen' in the name. Please set RUN_ID manually.")
    
    print(f"Found {len(runs)} 'unfrozen' runs (most recent first):\n")
    for idx, row in runs.iterrows():
        run_id = row['run_id']
        run_name = row.get('tags.mlflow.runName', 'N/A')
        start_time = row['start_time']
        status = row['status']
        num_labels = row.get('params.num_labels', 'N/A')
        mean_iou = row.get('metrics.final_mean_iou', 'N/A')
        print(f"  {idx+1}. {run_name}")
        print(f"     Run ID: {run_id}")
        print(f"     Status: {status}")
        print(f"     Started: {start_time}")
        print(f"     Classes: {num_labels}")
        print(f"     Final mIoU: {mean_iou}")
        print()
    
    # Use the most recent run
    RUN_ID = runs.iloc[0]['run_id']
    print(f"\n✅ Will use most recent run: {RUN_ID}")
    print(f"   If this is not the correct run, stop here and set RUN_ID manually in Cell 0.")
else:
    print(f"Using specified RUN_ID: {RUN_ID}")

Found 4 'unfrozen' runs (most recent first):

  1. unfrozen-1328class-20260311-1909
     Run ID: 6cc49e1ccb0d4b30b371e9a071dcbe6f
     Status: FINISHED
     Started: 2026-03-11 19:09:15.966000+00:00
     Classes: 1328
     Final mIoU: 0.6878568325225564

  2. unfrozen-1328class-20260311-1828
     Run ID: 4515c3ecde3a42f99de4f94252549ea0
     Status: RUNNING
     Started: 2026-03-11 18:28:15.906000+00:00
     Classes: 1328
     Final mIoU: nan

  3. unfrozen-1328class-20260311-1706
     Run ID: 4a6b1c6a363646009a0411056cfd3620
     Status: RUNNING
     Started: 2026-03-11 17:06:34.496000+00:00
     Classes: 1328
     Final mIoU: nan

  4. unfrozen-1328class-20260311-1538
     Run ID: 3d4e8ca618854418a2a1c2f70daf5f98
     Status: RUNNING
     Started: 2026-03-11 15:38:42.765000+00:00
     Classes: 1328
     Final mIoU: nan


✅ Will use most recent run: 6cc49e1ccb0d4b30b371e9a071dcbe6f
   If this is not the correct run, stop here and set RUN_ID manually in Cell 0.


In [0]:
# Cell 4 - Load the model and processor

from transformers import AutoImageProcessor, UperNetForSemanticSegmentation
import os

print("Loading model from DBFS...")

# Load model
model = UperNetForSemanticSegmentation.from_pretrained(MODEL_PATH)
print(f"✅ Loaded model")

# Check if processor was saved with the model
processor_path = os.path.join(MODEL_PATH, "preprocessor_config.json")
if os.path.exists(processor_path):
    # Load processor from saved model
    processor = AutoImageProcessor.from_pretrained(MODEL_PATH)
    print(f"✅ Loaded image processor from saved model")
else:
    # Processor wasn't saved - load from base DINOv2 model
    print(f"⚠️  No preprocessor_config.json found in model directory")
    print(f"   Loading processor from base facebook/dinov2-large model...")
    
    # Load from HuggingFace (or JFrog mirror if configured)
    HF_ENDPOINT = os.environ.get(
        "HF_ENDPOINT",
        "https://graingerreadonly.jfrog.io/artifactory/api/huggingfaceml/huggingfaceml-remote",
    )
    
    try:
        # Try JFrog first
        from huggingface_hub import hf_hub_download
        config_file = hf_hub_download(
            repo_id="facebook/dinov2-large",
            filename="preprocessor_config.json",
            endpoint=HF_ENDPOINT,
            cache_dir="/tmp/dinov2-processor"
        )
        processor = AutoImageProcessor.from_pretrained(os.path.dirname(config_file))
        print(f"✅ Loaded processor from JFrog mirror")
    except Exception as e:
        # Fallback to HuggingFace directly
        print(f"   JFrog failed, trying HuggingFace directly...")
        processor = AutoImageProcessor.from_pretrained("facebook/dinov2-large")
        print(f"✅ Loaded processor from HuggingFace")
    
    # Save processor to model directory for future use
    processor.save_pretrained(MODEL_PATH)
    print(f"   Saved processor to {MODEL_PATH}")

# Print model info
num_params = sum(p.numel() for p in model.parameters())
num_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nModel Summary:")
print(f"  Total parameters: {num_params:,}")
print(f"  Trainable parameters: {num_trainable:,}")
print(f"  Number of labels: {model.config.num_labels}")
print(f"  Model type: {model.config.model_type}")

2026-03-12 12:40:39.024908: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773319239.040704    4508 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773319239.045569    4508 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773319239.058954    4508 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773319239.058966    4508 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773319239.058968    4508 computation_placer.cc:177] computation placer alr

[2026-03-12 12:40:41,566] [INFO] [real_accelerator.py:239:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -lcufile: No such file or directory
collect2: error: ld returned 1 exit status


Loading model from DBFS...
✅ Loaded model
⚠️  No preprocessor_config.json found in model directory
   Loading processor from base facebook/dinov2-large model...


(…)ge/resolve/main/preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


✅ Loaded processor from JFrog mirror
   Saved processor to /dbfs/FileStore/allen_brain_data/models/unfrozen

Model Summary:
  Total parameters: 342,104,160
  Trainable parameters: 342,104,160
  Number of labels: 1328
  Model type: upernet


In [0]:
# Cell 4 - Load the model

from transformers import AutoImageProcessor, UperNetForSemanticSegmentation

print("Loading model from DBFS...")

# Load processor (image preprocessor)
processor = AutoImageProcessor.from_pretrained(MODEL_PATH)
print(f"✅ Loaded image processor")

# Load model
model = UperNetForSemanticSegmentation.from_pretrained(MODEL_PATH)
print(f"✅ Loaded model")

# Print model info
num_params = sum(p.numel() for p in model.parameters())
num_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nModel Summary:")
print(f"  Total parameters: {num_params:,}")
print(f"  Trainable parameters: {num_trainable:,}")
print(f"  Number of labels: {model.config.num_labels}")
print(f"  Model type: {model.config.model_type}")

Loading model from DBFS...
✅ Loaded image processor
✅ Loaded model

Model Summary:
  Total parameters: 342,104,160
  Trainable parameters: 342,104,160
  Number of labels: 1328
  Model type: upernet


In [0]:
# Cell 5 - Log model to MLflow

import mlflow.transformers

print(f"Logging model to MLflow run {RUN_ID}...")
print(f"This may take several minutes for a large model.\n")

# Resume the existing run to add artifacts
with mlflow.start_run(run_id=RUN_ID) as run:
    # Log the model with both the model and processor
    mlflow.transformers.log_model(
        transformers_model={
            "model": model,
            "image_processor": processor
        },
        artifact_path="model",
        registered_model_name="dinov2-upernet-unfrozen",
        task="image-segmentation",
        metadata={
            "source": "retroactive_log",
            "original_path": MODEL_PATH,
            "num_labels": model.config.num_labels,
            "mapping_type": "full"
        }
    )
    
    print(f"✅ Model logged successfully!")
    print(f"\nArtifact path: model")
    print(f"Registered model name: dinov2-upernet-unfrozen")
    
    # Also log the local DBFS path as a tag for reference
    mlflow.set_tag("dbfs_model_path", MODEL_PATH)
    print(f"\nTagged with DBFS path: {MODEL_PATH}")

Logging model to MLflow run 6cc49e1ccb0d4b30b371e9a071dcbe6f...
This may take several minutes for a large model.



2026/03/12 12:41:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://grainger-gtg-mlops-dev-use1-dbx-shared-ws.cloud.databricks.com/ml/experiments/1345391216675532/models/m-97f5b15d98f549788b8f04033dffe1f9?o=2648165776397546
Device set to use cuda:0
2026/03/12 12:43:02 WARNING mlflow.transformers: The model card could not be retrieved from the hub due to Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/dbfs/FileStore/allen_brain_data/models/unfrozen'. Use `repo_type` argument if needed.
2026/03/12 12:43:02 WARNING mlflow.transformers: Unable to find license information for this model. Please verify permissible usage for the model you are storing prior to use.
2026/03/12 12:43:02 WARNING mlflow.transformers: This model is unable to be used for pyfunc prediction because the model is not a language-based model and requires a complex input type that is currently not supported. The pyfunc flavor will no

---------------------------------------------------------------------------
RestException                             Traceback (most recent call last)
File /databricks/python/lib/python3.12/site-packages/mlflow/store/_unity_catalog/registry/rest_store.py:361, in UcModelRegistryStore.create_registered_model(self, name, tags, description, deployment_job_id)
    360 try:
--> 361     response_proto = self._call_endpoint(CreateRegisteredModelRequest, req_body)
    362 except RestException as e:

File /databricks/python/lib/python3.12/site-packages/mlflow/store/model_registry/base_rest_store.py:42, in BaseRestStore._call_endpoint(self, api, json_body, call_all_endpoints, extra_headers)
     41 endpoint, method = self._get_endpoint_from_method(api)
---> 42 return call_endpoint(
     43     self.get_host_creds(), endpoint, method, json_body, response_proto, extra_headers
     44 )

File /databricks/python/lib/python3.12/site-packages/mlflow/utils/rest_utils.py:590, in call_endpoint(host_creds

In [0]:
# Cell 6 - Verify the model was logged

# Refresh run info
run = mlflow.get_run(RUN_ID)

# List artifacts
artifacts = mlflow.artifacts.list_artifacts(run_id=RUN_ID)

print(f"Artifacts in run {RUN_ID}:")
for artifact in artifacts:
    print(f"  - {artifact.path}")
    if artifact.is_dir:
        # List contents of directories
        sub_artifacts = mlflow.artifacts.list_artifacts(run_id=RUN_ID, artifact_path=artifact.path)
        for sub in sub_artifacts[:5]:  # Show first 5 items
            print(f"    - {sub.path}")
        if len(sub_artifacts) > 5:
            print(f"    ... and {len(sub_artifacts) - 5} more")

print(f"\n✅ SUCCESS! Model is now available in MLflow.")
print(f"\nYou can now:")
print(f"  1. View the model in MLflow UI")
print(f"  2. Load it with: mlflow.transformers.load_model('runs:/{RUN_ID}/model')")
print(f"  3. Use the registered model: mlflow.transformers.load_model('models:/dinov2-upernet-unfrozen/latest')")

Artifacts in run 6cc49e1ccb0d4b30b371e9a071dcbe6f:

✅ SUCCESS! Model is now available in MLflow.

You can now:
  1. View the model in MLflow UI
  2. Load it with: mlflow.transformers.load_model('runs:/6cc49e1ccb0d4b30b371e9a071dcbe6f/model')
  3. Use the registered model: mlflow.transformers.load_model('models:/dinov2-upernet-unfrozen/latest')


## Summary

This notebook successfully loaded the trained model from DBFS and logged it to MLflow.

**Next Steps:**
1. Update your training notebooks to include `mlflow.transformers.log_model()` before `mlflow.end_run()`
2. Consider creating a helper function to log models consistently across all experiments
3. Document this as a lesson learned

**Prevention:**
Add this code BEFORE `mlflow.end_run()` in future training notebooks:

```python
# Log model to MLflow artifacts
import mlflow.transformers
mlflow.transformers.log_model(
    transformers_model={
        "model": trainer.model,
        "image_processor": processor
    },
    artifact_path="model",
    registered_model_name="your-model-name",
    task="image-segmentation"
)
```